# Document Retrieval Implementation: Movie Search Engine

Problem Statement: Build a Basic Search Engine that retrieves the most relevant movie descriptions from a dataset of 34,000+ Wikipedia movie plots.

In [ ]:
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Load Data

In [ ]:
import pandas as pd

drive_path = '/content/drive/My Drive/Deep Learning Content/NLP/NLP Colab Notebooks/Datasets/Document Retrieval/wiki_movie_plots_deduped.csv'
df = pd.read_csv(drive_path)
df.head(10)

#df=pd.read_csv("wiki_movie_plots_deduped.csv")
#df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...
5,1903,Alice in Wonderland,American,Cecil Hepworth,May Clark,unknown,https://en.wikipedia.org/wiki/Alice_in_Wonderl...,"Alice follows a large white rabbit down a ""Rab..."
6,1903,The Great Train Robbery,American,Edwin S. Porter,NaN,western,https://en.wikipedia.org/wiki/The_Great_Train_...,The film opens with two bandits breaking into ...
7,1904,The Suburbanite,American,Wallace McCutcheon,NaN,comedy,https://en.wikipedia.org/wiki/The_Suburbanite,The film is about a family who move to the sub...
8,1905,The Little Train Robbery,American,Edwin Stanton Porter,NaN,unknown,https://en.wikipedia.org/wiki/The_Little_Train...,The opening scene shows the interior of the ro...
9,1905,The Night Before Christmas,American,Edwin Stanton Porter,NaN,unknown,https://en.wikipedia.org/wiki/The_Night_Before...,Scenes are introduced using lines of the poem....


We will mainly use the Plot column for building our search engine.

In [ ]:
df['Plot'].isna()

,Plot
0,False
1,False
2,False
3,False
4,False
...,...
34881,False
34882,False
34883,False
34884,False


In [ ]:
df[df['Plot'].isna()]

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot


In [ ]:
plots = df['Plot'].fillna("").tolist() # Replaces any NaN (missing) values in the 'Plot' column with an empty string "".
print("Total Number of Plots:", len(plots))

Total Number of Plots: 34886


## Build TF-IDF Vectorizer

We create a TF-IDF vectorizer to convert text into numerical vectors.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words='english',
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(plots)
print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (34886, 50000)


## Define Search Function

The `search_movies()` function:

Converts the query to a TF-IDF vector

Computes similarity between query and all movie plots

Returns the Top-K most relevant movies

Each result contains Rank, Score, Title, Genre, Year, and a Plot preview.

In [ ]:
def search_movies(query, top_k=5):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]

    results = []
    for i in top_idx:
        results.append({
            "rank": len(results)+1,
            "score": round(scores[i], 4),
            "title": df.iloc[i]["Title"],
            "year": df.iloc[i]["Release Year"],
            "genre": df.iloc[i]["Genre"],
            "plot": df.iloc[i]["Plot"][:500] + "..."
        })
    return results

## Display Function

This function prints each search result in a clean, readable format for students.

In [ ]:
def display_results(results):
    for r in results:
        print("="*80)
        print(f"Rank: {r['rank']}")
        print(f"Score: {r['score']}")
        print(f"Title: {r['title']} | {r['year']}")
        print(f"Genre: {r['genre']}")
        print(f"Plot: {r['plot']}")

## Test Queries

In [ ]:
results = search_movies("artificial intelligence robot destroys humanity", top_k=5)
display_results(results)

Rank: 1
Score: 0.3057
Title: Armed Response | 2017
Genre: thriller, sci-fi, action
Plot: A team of trained operatives find themselves trapped inside an isolated military compound after its artificial intelligence is suddenly shut down. There, they begin to experience strange and horrific phenomena as they attempt to uncover what killed the previous team....
Rank: 2
Score: 0.2959
Title: Daft Punk's Electroma | 2006
Genre: science fiction
Plot: The two lead characters appear as the robotic forms of Daft Punk and are credited as "Hero Robot No. 1" and "Hero Robot No. 2". One wears a silver helmet and the other wears a golden one. An opening scene shows the duo driving in a 1987 Ferrari 412 with its license plate displaying "HUMAN". After passing through a Southwestern United States landscape, the duo arrives by car at a town in Inyo County, California.[1] The town's resident are also shown to be robots physically identical to the two ma...
Rank: 3
Score: 0.2796
Title: Solar Crisis | 1990


In [ ]:
results = search_movies("time travel science fiction thriller", top_k=5)
display_results(results)

Rank: 1
Score: 0.4105
Title: Chhalia | 1973
Genre: action
Plot: Chhalia is a family thriller....
Rank: 2
Score: 0.2969
Title: Haathkadi | 1982
Genre: family, thriller, drama
Plot: Haathkadi is a suspense thriller [1]...
Rank: 3
Score: 0.2871
Title: Remote Control | 1988
Genre: comedy
Plot: A video store clerk stumbles onto an alien plot to take over earth by brainwashing people with a bad '50s science fiction movie. He and his friends race to stop the aliens before the tapes can be distributed world-wide....
Rank: 4
Score: 0.2752
Title: Hullabaloo | 1940
Genre: musical comedy
Plot: Morgan is the star of the film, as a fading actor Frankie Merriweather who is trying to revive his career by starring on a radio program. When his most recent broadcast, a science fiction invasion from Mars story, panics the nation, he is fired. He decides to jumpstart his career by creating a new show which features his talented children....
Rank: 5
Score: 0.2661
Title: Orders Are Orders | 1954
Genre: comed

In [ ]:
results = search_movies("comedy and horror", top_k=5)
display_results(results)

Rank: 1
Score: 0.442
Title: I Was a Teenage Zombie | 1987
Genre: horror comedy
Plot: The film begins like a 1980s comedy with teens looking to purchase some marijuana but turns into comedy/horror genre when a drug dealer is pushed into the river and becomes a zombie....
Rank: 2
Score: 0.2952
Title: The Moon's Our Home | 1936
Genre: comedy
Plot: A comedy about marriage and everything relating to it. New York novelist Henry Fonda meets up with an actress, Margaret Sullavan, and the two date and later marry, though neither knows of the other's fame. The real adventure begins on the honeymoon, when this screwball comedy really heats up with insults and arguments....
Rank: 3
Score: 0.295
Title: Yellow Sands | 1938
Genre: comedy/drama
Plot: A rural comedy about a rich dying woman's relatives that are about to be disappointed by the contents of her will....
Rank: 4
Score: 0.2763
Title: Min & Max | 2016
Genre: comedy / romance
Plot: The comedy and love story between a man and a women with heig

In [ ]:
results = search_movies("comedy/drama", top_k=5)
display_results(results)

Rank: 1
Score: 0.4771
Title: Lal Dupatta Malmal Ka | 1989
Genre: unknown
Plot: This is a romantic drama movie.[2]...
Rank: 2
Score: 0.4021
Title: Nothing Else Matters | 1920
Genre: comedy
Plot: A comedy/drama genre film, about the life of a British Music Hall comic....
Rank: 3
Score: 0.3058
Title: I Was a Teenage Zombie | 1987
Genre: horror comedy
Plot: The film begins like a 1980s comedy with teens looking to purchase some marijuana but turns into comedy/horror genre when a drug dealer is pushed into the river and becomes a zombie....
Rank: 4
Score: 0.2904
Title: The Moon's Our Home | 1936
Genre: comedy
Plot: A comedy about marriage and everything relating to it. New York novelist Henry Fonda meets up with an actress, Margaret Sullavan, and the two date and later marry, though neither knows of the other's fame. The real adventure begins on the honeymoon, when this screwball comedy really heats up with insults and arguments....
Rank: 5
Score: 0.2902
Title: Yellow Sands | 1938
Genre: co

## Conclusion:
This Implementation demonstrated how a TF-IDF–based search engine retrieves movie plots using keyword similarity.
While effective for basic text matching, TF-IDF does not understand deeper meaning or context.
For improved performance, we can use semantic methods like Word2Vec, Doc2Vec, or BERT embeddings.


